# Hybrid Retrieval with Pinecone + LangChain (BM25 + Dense + Rerank)

Rebuild of [daveebbelaar's hybrid-retrieval tutorial](https://github.com/daveebbelaar/ai-cookbook/tree/main/knowledge/hybrid-retrieval), using **Pinecone** as the vector database and **LangChain** as the framework.

**Stack for this version:**
- **Sparse:** `pinecone_text.sparse.BM25Encoder`, fit on our own corpus (LangChain has no BM25 of its own — the retriever expects this exact object)
- **Dense:** `langchain_huggingface.HuggingFaceEmbeddings` wrapping `all-MiniLM-L6-v2` (local, free, 384-dim)
- **Storage + fusion:** `langchain_community.retrievers.PineconeHybridSearchRetriever` — one Pinecone index (`metric="dotproduct"`), alpha-weighted hybrid search built into the retriever
- **Rerank:** `langchain_pinecone.PineconeRerank` (bge-reranker-v2-m3) wrapped in `ContextualCompressionRetriever`

**Dataset:** [FiQA-2018](https://sites.google.com/view/fiqa) — 57,638 financial forum posts, 648 labeled test questions.

### The key conceptual difference from what you've learned (RRF)

The original tutorial ran BM25 and dense search separately and fused the **ranks** with Reciprocal Rank Fusion: `score(d) = sum(1 / (k + rank))`.

`PineconeHybridSearchRetriever` fuses **vectors**, not rankings. Every document has one record with both a dense and a sparse vector. At query time:

```
score = alpha * dense_score + (1 - alpha) * sparse_score
```

`alpha` is just a constructor argument on the retriever (`alpha=1.0` → dense only, `alpha=0.0` → sparse only, `alpha=0.5` → balanced). LangChain hides the vector math, but it's the same alpha-scaling mechanism under the hood — genuinely different from RRF, not just a different library wrapping the same idea.

## Setup

```bash
pip install langchain langchain-community langchain-huggingface langchain-pinecone \
            pinecone pinecone-text sentence-transformers datasets pandas numpy tqdm python-dotenv
```

One API key: `PINECONE_API_KEY`, free at [app.pinecone.io](https://app.pinecone.io). No OpenAI or Cohere key needed.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

DATA_DIR = Path("data/fiqa")
INDEX_DIR = Path("indexes")
INDEX_DIR.mkdir(exist_ok=True)

assert os.getenv("PINECONE_API_KEY"), "Set PINECONE_API_KEY in your .env file"
print("Pinecone API key loaded.")

## Step 1 — Download and explore the data

Every BEIR dataset ships the same shape: a **corpus** (documents), **queries** (user questions), and **qrels** (ground-truth query→relevant-doc judgments).

In [ ]:
from datasets import load_dataset

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "corpus.parquet").exists():
    corpus_ds = load_dataset("BeIR/fiqa", "corpus", split="corpus")
    queries_ds = load_dataset("BeIR/fiqa", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/fiqa-qrels", split="test")

    corpus_ds.to_parquet(DATA_DIR / "corpus.parquet")
    queries_ds.to_parquet(DATA_DIR / "queries.parquet")
    qrels_ds.to_parquet(DATA_DIR / "qrels.parquet")
    print("Downloaded and cached FiQA as parquet.")
else:
    print("FiQA already cached locally.")

In [ ]:
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
qrels = pd.read_parquet(DATA_DIR / "qrels.parquet")

doc_ids = corpus["_id"].tolist()
doc_texts = corpus["text"].tolist()

print(f"Corpus:  {len(corpus):>6} documents")
print(f"Queries: {len(queries):>6} total")
print(f"Qrels:   {len(qrels):>6} relevance judgments")
print("\nExample document:")
print(doc_texts[0][:300])

In [ ]:
test_query_ids = set(qrels["query-id"].astype(str))
test_queries = queries[queries["_id"].astype(str).isin(test_query_ids)]
print(f"Test queries (with qrels): {len(test_queries)}")

example_query = test_queries.iloc[0]
print(f"\nExample query: {example_query['text']}")

relevant = qrels[qrels["query-id"].astype(str) == str(example_query["_id"])]
print(f"Relevant doc ids: {list(relevant['corpus-id'])}")

## Step 2 — Sparse encoding with BM25

`PineconeHybridSearchRetriever` doesn't ship its own BM25 implementation — it takes a `sparse_encoder` object with `.encode_documents()` / `.encode_queries()`, and every LangChain + Pinecone hybrid search example uses `pinecone_text.sparse.BM25Encoder` for that. We fit it on our own corpus (not `.default()`, which ships MS MARCO statistics) so term/document-frequency numbers reflect FiQA's vocabulary.

In [ ]:
from pinecone_text.sparse import BM25Encoder

bm25 = BM25Encoder()
bm25.fit(doc_texts)

BM25_PARAMS_PATH = INDEX_DIR / "bm25_params.json"
bm25.dump(str(BM25_PARAMS_PATH))
print(f"Fit BM25 on {len(doc_texts)} docs, saved params to {BM25_PARAMS_PATH}")

In [ ]:
# Sanity check
doc_sparse = bm25.encode_documents(doc_texts[0])
print(f"Doc sparse vector: {len(doc_sparse['indices'])} non-zero terms")

query_sparse = bm25.encode_queries("Where should I park my rainy-day fund?")
print(f"Query sparse vector: {len(query_sparse['indices'])} non-zero terms")

## Step 3 — Dense embeddings via LangChain

`HuggingFaceEmbeddings` wraps `sentence-transformers/all-MiniLM-L6-v2` behind LangChain's standard `Embeddings` interface (`embed_documents` / `embed_query`). It runs locally — no OpenAI key, no per-token cost. We normalize so dotproduct == cosine similarity, which matters because Pinecone's hybrid index requires `metric="dotproduct"`.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

DENSE_DIM = 384  # all-MiniLM-L6-v2 output size -- must match the Pinecone index dimension

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# quick check -- embed_query returns a plain python list of floats
sample_vec = embeddings.embed_query("test sentence")
print(f"Embedding dim: {len(sample_vec)}")

## Step 4 — Create the Pinecone hybrid index

Same two hard requirements as the raw-SDK version:
- the index metric **must** be `"dotproduct"`
- every record with sparse values must also carry dense values

The difference: we won't call `index.upsert()` ourselves. `PineconeHybridSearchRetriever.add_texts()` (next cell) does that for us.

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
INDEX_NAME = "fiqa-hybrid-lc"

existing = [idx["name"] for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DENSE_DIM,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"Created index '{INDEX_NAME}'")
else:
    print(f"Index '{INDEX_NAME}' already exists, reusing it")

index = pc.Index(INDEX_NAME)

## Step 5 — Build the retriever and load the corpus

`PineconeHybridSearchRetriever(embeddings=..., sparse_encoder=..., index=..., alpha=...)` is the object that replaces the raw-SDK version's `hybrid_scale()` function, `upsert_batch()`, and `hybrid_query()` all at once.

`retriever.add_texts()` embeds every document (dense), sparse-encodes it (BM25), and upserts both into the index — one call instead of two separate scripts. We pass explicit `ids` (the FiQA doc ids) and mirror them into `metadatas` so we can map search results back to ground truth later — the retriever's returned `Document` objects carry whatever's in `metadata`, but not the raw Pinecone vector id.

In [ ]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

# alpha=0.5 here is just this instance's default -- we'll build other instances
# with different alpha in Step 6 to compare sparse-only / dense-only / hybrid.
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    alpha=0.5,
    top_k=10,
)

BATCH = 200
if index.describe_index_stats()["total_vector_count"] < len(doc_ids):
    for i in tqdm(range(0, len(doc_ids), BATCH), desc="add_texts"):
        batch_ids = doc_ids[i : i + BATCH]
        batch_texts = doc_texts[i : i + BATCH]
        retriever.add_texts(
            texts=batch_texts,
            ids=batch_ids,
            metadatas=[{"doc_id": d} for d in batch_ids],
        )
    print(index.describe_index_stats())
else:
    print("Index already populated, skipping add_texts.")

## Step 6 — Hybrid search: compare alpha values

Each `PineconeHybridSearchRetriever` instance is a fixed `(alpha, top_k)` view over the same index. Building one is free (no network call) — only `.invoke()` hits Pinecone. So to compare sparse-only vs dense-only vs hybrid, we just build three instances.

In [ ]:
sparse_retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.0, top_k=5)
dense_retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=1.0, top_k=5)
hybrid_retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.5, top_k=5)


def show(label, docs):
    print(f"\n{label}")
    for i, doc in enumerate(docs, 1):
        score = doc.metadata.get("score")
        doc_id = doc.metadata.get("doc_id")
        score_str = f"{score:.4f}" if score is not None else "  n/a "
        print(f"  {i}. [{score_str}] {doc_id}  {doc.page_content[:70]}")


query = "Where should I park my rainy-day fund?"
print(f"Query: {query}")

show("Sparse only (alpha=0.0)", sparse_retriever.invoke(query))
show("Dense only (alpha=1.0)", dense_retriever.invoke(query))
show("Hybrid (alpha=0.5)", hybrid_retriever.invoke(query))

## Step 7 — Rerank with ContextualCompressionRetriever

A bi-encoder (our dense retriever) embeds query and document separately. A **cross-encoder** feeds both into the same model together for a joint relevance score — slower, but catches subtleties a bi-encoder misses.

`PineconeRerank` is a LangChain **document compressor**: it takes a list of `Document`s + a query and returns a smaller, reordered list. Wrapping a retriever in `ContextualCompressionRetriever` gives you one retriever object that does retrieve-then-rerank in a single `.invoke()` — the two-stage pattern (top-50 hybrid candidates → cross-encoder → top-10) expressed as retriever composition instead of manually chaining function calls.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_pinecone import PineconeRerank

hybrid_retriever_50 = PineconeHybridSearchRetriever(
    embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.5, top_k=50
)

reranker = PineconeRerank(model="bge-reranker-v2-m3", top_n=10)

reranked_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever_50,
)


def show_reranked(label, docs):
    print(f"\n{label}")
    for i, doc in enumerate(docs[:5], 1):
        score = doc.metadata.get("relevance_score", doc.metadata.get("score"))
        doc_id = doc.metadata.get("doc_id")
        print(f"  {i}. [{score:.4f}] {doc_id}  {doc.page_content[:70]}")


show_reranked("Hybrid (alpha=0.5) only", hybrid_retriever_50.invoke(query))
show_reranked("Hybrid + Pinecone rerank (bge-reranker-v2-m3)", reranked_retriever.invoke(query))

## Step 8 — Evaluate with NDCG@10

Same metric as the original tutorial: rewards relevant docs ranked high in the top-10, perfect ranking = 1.0. We sample a subset of test queries to stay fast and within the free rerank quota.

In [ ]:
import math
from collections import defaultdict

SAMPLE_SIZE = 50
SEED = 42


def ndcg_at_k(predicted_ids, relevant: dict, k: int = 10) -> float:
    dcg = sum(
        relevant.get(doc_id, 0) / math.log2(rank + 2)
        for rank, doc_id in enumerate(predicted_ids[:k])
    )
    ideal_rels = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0.0


qrels_map = defaultdict(dict)
for _, row in qrels.iterrows():
    qrels_map[str(row["query-id"])][str(row["corpus-id"])] = int(row["score"])

sample = test_queries.sample(n=SAMPLE_SIZE, random_state=SEED)
print(f"Evaluating on {len(sample)} queries")


def ids_from(docs):
    return [d.metadata["doc_id"] for d in docs]

In [ ]:
eval_results = defaultdict(list)

sparse_r10 = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.0, top_k=10)
dense_r10 = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=1.0, top_k=10)
hybrid_r10 = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25, index=index, alpha=0.5, top_k=10)

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Evaluating"):
    query_id = str(row["_id"])
    query_text = row["text"]
    relevant = qrels_map[query_id]

    eval_results["Sparse (BM25)"].append(ndcg_at_k(ids_from(sparse_r10.invoke(query_text)), relevant))
    eval_results["Dense (all-MiniLM-L6-v2)"].append(ndcg_at_k(ids_from(dense_r10.invoke(query_text)), relevant))
    eval_results["Hybrid (alpha=0.5)"].append(ndcg_at_k(ids_from(hybrid_r10.invoke(query_text)), relevant))
    eval_results["Hybrid + Rerank"].append(ndcg_at_k(ids_from(reranked_retriever.invoke(query_text)), relevant))

print(f"\nNDCG@10 x100 on FiQA ({len(sample)} sampled test queries)")
print("-" * 42)
for method, scores in eval_results.items():
    print(f"  {method:<26} {np.mean(scores) * 100:.2f}")

## Wrap-up

- **Expect the dense-only row to land a bit under the original tutorial's ~31** — `all-MiniLM-L6-v2` is a much smaller model than `text-embedding-3-small`. The hybrid + rerank stages exist to close that gap.
- **Every retriever is a cheap, disposable object.** No connection pooling or state to manage — construct with the alpha/top_k you need, call `.invoke()`, done.
- **The `scripts/` folder** breaks this into numbered `.py` files (`1_explore_data.py` → `6_evaluate.py`) plus a shared `utils/langchain_setup.py`, same split as this notebook, so you have both the "see everything inline" version and the "import from modules" version.
- **To point this at your own data:** replace the FiQA corpus with your own `(_id, text)` parquet, skip the download cell, and rerun from Step 2.